# Precondition Validity

Pass rate alone can be misleading when generated tests do not satisfy the requirement precondition. This notebook computes validity-aware failure rates from the paper-derived requirements table.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/casparbreloh/rbt4dnn-seminar.git"
COLAB_ROOT = Path("/content/rbt4dnn-seminar")
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "data/requirements.csv").exists()), None)
if ROOT is None:
    ROOT = COLAB_ROOT
    if (ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"], check=True)
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)

for module in [
    "shared",
    "mnist",
    "precondition_validity",
    "cost_analysis",
    "mnist_lora_ablation",
    "shared_lora_pilot",
]:
    sys.modules.pop(module, None)

EXPERIMENT = ROOT / "experiments" / "precondition-validity"
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(EXPERIMENT))
print(ROOT)
commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "log", "-1", "--oneline"], text=True
).strip()
print(commit)

In [ ]:
from precondition_validity import build_rows, write_results
from shared import requirement_rows

results_path, summary_path = write_results(ROOT)
rows = build_rows(requirement_rows(ROOT))

print("highest estimated valid-failure rates")
for row in rows[:10]:
    print(
        f"{row['dataset']} {row['requirement']} {row['method']} | "
        f"estimated_valid_fail_rate={row['estimated_valid_failure_rate']} pass={row['pass_rate']} pre={row['precondition_match']}"
    )
print(results_path)
print(summary_path)

The output files are `results.csv` and `summary.md`.